In [1]:
import anatomist.api as ana
from soma.qt_gui.qtThread import QtThreadCall
from soma.qt_gui.qt_backend import Qt

a = ana.Anatomist()

from soma import aims
import pandas as pd
import numpy as np
import os

/usr/lib/python3/dist-packages/scipy/__init__.py:146: UserWarning: A NumPy version >=1.17.3 and <1.25.0 is required for this version of SciPy (detected version 1.26.4
  warnings.warn(f"A NumPy version >={np_minversion} and <{np_maxversion}"
existing QApplication: 0
QStandardPaths: XDG_RUNTIME_DIR not set, defaulting to '/tmp/runtime-ad279118'


create qapp
done
Starting Anatomist.....
config file : /casa/home/.anatomist/config/settings.cfg
PyAnatomist Module present
PythonLauncher::runModules()
global modules: /casa/host/build/share/anatomist-5.2/python_plugins
home   modules: /casa/home/.anatomist/python_plugins
loading module simple_controls
loading module save_resampled
loading module selection
loading module bsa_proba
loading module modelGraphs
loading module profilewindow
loading module ana_image_math
loading module paletteViewer
loading module foldsplit
loading module anacontrolmenu
loading module gradientpalette
loading module palettecontrols
loading module meshsplit
loading module volumepalettes
loading module gltf_io
loading module infowindow
loading module histogram
loading module measure
loading module statsplotwindow
loading module valuesplotwindow
all python modules loaded
Anatomist started.


#### To visualize specific 3D volumic sulci for specific subjects

In [2]:
dataset = 'UkBioBank40'# 'UkBioBank40' 'hcp
region = "CINGULATE." #"S.C.-sylv." "S.T.s." "CINGULATE."
side = 'R' #"L"

In [ ]:
#sorted_phenotype = pd.read_csv('/volatile/ad279118/Imaging_Genetics_2025/UKB_cingulate_pred/UKB_left_pred.csv')
#sorted_phenotype = sorted_phenotype.sort_values(by='Left_Prob_Pred')
#sorted_phenotype.IID = sorted_phenotype['IID'].apply(lambda x : 'sub-'+str(x))
#list_subjects = sorted_phenotype['IID'].to_list()

my_labelled_df = pd.read_csv('/neurospin/dico/data/deep_folding/current/datasets/UkBioBank40/CSLabel/interrupted_CS_QC.csv')
#my_labelled_df = pd.read_csv('/volatile/ad279118/UKB/CentralSulcus/interruption_pred.csv')
sample = my_labelled_df[my_labelled_df.Note=='Very short C.S.'].ID.iloc[0:2]
my_labelled_df.Note.unique()
print(sample.to_list())
sample = pd.read_csv("/neurospin/dico/data/deep_folding/current/datasets/UkBioBank40/skeletons/skeleton_outliers.csv")
sample = sample['ID'].to_list()

def f_id(x):
    return "sub-"+str(x)

In [3]:
sample = pd.read_csv("/neurospin/dico/data/deep_folding/current/datasets/UkBioBank40/PCS_annotation/100_SEX_AGE_strat.csv")
sample = sample['Subject'].to_list()
sample = sample[35:40]

In [26]:
#dataframe = pd.DataFrame({'ID':sub, 'Note':QC})
#dataframe.Note.unique()
#dataframe.to_csv('/neurospin/dico/data/deep_folding/current/datasets/UkBioBank40/CSLabel/interrupted_CS_QC.csv', index=False)

In [ ]:
volume=True
nb_columns=2
block = a.createWindowsBlock(nb_columns) # nb of columns
dic_windows = {}

referential1 = a.createReferential()

mm_skeleton_path = f'/neurospin/dico/data/deep_folding/current/datasets/{dataset}/crops/2mm/{region}/mask/{side}crops'
dic_windows['Sulci_color']=a.loadObject('/casa/host/build/share/brainvisa-share-5.2/nomenclature/hierarchy/sulcal_root_colors.hie')
for i, subject_id in enumerate(sample):
    volume_path = f"{mm_skeleton_path}/{subject_id}_cropped_skeleton.nii.gz"
    
    if os.path.isfile(volume_path):
        vol = aims.read(volume_path)
        
        dic_windows[f'a_vol{nb_columns*i}'] = a.toAObject(vol)
        #dic_windows[f'a_vol{i}'].setPalette(absoluteMode=True)
        dic_windows[f'rvol{nb_columns*i}'] = a.fusionObjects(objects=[dic_windows[f'a_vol{nb_columns*i}']], method='VolumeRenderingFusionMethod')
        dic_windows[f'rvol{nb_columns*i}'].releaseAppRef()
        dic_windows[f'rvol{nb_columns*i}'].assignReferential(referential1)
        dic_windows[f'wvr{nb_columns*i}'] = a.createWindow('3D', block=block) #geometry=[100+400*(i%3), 100+440*(i//3), 400, 400])
        dic_windows[f'wvr{nb_columns*i}'].addObjects(dic_windows[f'rvol{nb_columns*i}'])
    else:
        print(f"{volume_path} is not a correct path, or the .nii.gz doesn't exist")

    path_to_t1mri = f'/home/ad279118/tmp1/{subject_id}/ses-2/anat/t1mri/default_acquisition'
    white_matter_path = f'{path_to_t1mri}/default_analysis/segmentation/mesh/{subject_id}_{side}white.gii'
    sulci_path = f'{path_to_t1mri}/default_analysis/folds/3.1/{side}{subject_id}.arg'
    spam_labelled_sulci_path = f'{path_to_t1mri}/default_analysis/folds/3.1/spam_session_auto/{side}{subject_id}_spam_session_auto.arg'
    deep_labelled_sulci_path = f'{path_to_t1mri}/default_analysis/folds/3.1/deepcnn_session_auto/{side}{subject_id}_deepcnn_session_auto.arg'

    if os.path.isfile(white_matter_path):
        # To visualize the white matter for specific people
        dic_windows[f'white_{subject_id}'] = a.loadObject(white_matter_path)
        #dic_windows[f'white_{subject_id}'].loadReferentialFromHeader()
        dic_windows[f'white_{subject_id}'].assignReferential(referential1)
    else:
        print(f"{white_matter_path} is not a correct path, or the .white.gii doesn't exist")

    #if os.path.isfile(sulci_path):
        # To visualize the sulci for specific people
        #dic_windows[f'sulci_{subject_id}'] = a.loadObject(sulci_path)
        #dic_windows[f'sulci_{subject_id}'].loadReferentialFromHeader()
    #else:
        #print(f"{sulci_path} is not a correct path, or the .arg doesn't exist")
    
    if os.path.isfile(spam_labelled_sulci_path):
        # To visualize the annotated sulci for specific people
        dic_windows[f'sulci_labelled_{subject_id}'] = a.loadObject(spam_labelled_sulci_path)
        #dic_windows[f'sulci_labelled_{subject_id}'].loadReferentialFromHeader()
        dic_windows[f'sulci_labelled_{subject_id}'].assignReferential(referential1)
    else:
        print(f"{spam_labelled_sulci_path} is not a correct path, or the .arg doesn't exist")
        print("Automatic try with 'deepcnn_session_auto' instead of 'spam_session_auto'")
        if  os.path.isfile(deep_labelled_sulci_path):
            # To visualize the annotated sulci for specific people
            dic_windows[f'sulci_labelled_{subject_id}'] = a.loadObject(deep_labelled_sulci_path)
            #dic_windows[f'sulci_labelled_{subject_id}'].loadReferentialFromHeader()
            dic_windows[f'sulci_labelled_{subject_id}'].assignReferential(referential1)
    
    #dic_windows[f'wvr{nb_columns*i+1}'] = a.createWindow('3D', block=block)
    #dic_windows[f'wvr{nb_columns*i+1}'].addObjects([dic_windows2[f'white_{subject_id}'], dic_windows[f'sulci_{subject_id}']])
    dic_windows[f'wvr{nb_columns*i+1}'] = a.createWindow('3D', block=block)
    dic_windows[f'wvr{nb_columns*i+1}'].addObjects([dic_windows[f'white_{subject_id}'], dic_windows[f'sulci_labelled_{subject_id}']])

In [ ]:
#1;1#0;0.644444;1;1#0;0.622222;1;1#0;0;1;1
# 0;1;1;0#0;0.644444;1;0#0;0.622222;1;0#0;0;0.371795;1;0.658974;0.355556;1;0

#### To visualize the BUCKETS for specific people 

In [ ]:
bucket_path = f'/neurospin/dico/data/deep_folding/current/datasets/{dataset}/crops/2mm/{region}/mask/{side}buckets'

bucket_files = []
bck_path = f'{bucket_path}/{subject_id}_cropped_skeleton.bck'

for subject_id in sample:
    if os.path. isfile(bck_path):
        bucket_files.append(bck_path)
    else:
        print(f"{bck_path} is not a correct path, or the .bck doesn't exist")

for i, file in enumerate(bucket_files):
    dic_windows[f'bck_{i}'] = a.loadObject(file)
    dic_windows[f'w_{i}'] = a.createWindow('3D', block=block)#geometry=[100+400*(i%3), 100+440*(i//3), 400, 400])
    dic_windows[f'w_{i}'].addObjects(dic_windows[f'bck_{i}'])